In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Embedding + Positional Encoding

In [ ]:
class Embedding(nn.Module):
    def __init__(self , d_model , vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size , d_model)
    def forward(self , x):
        return self.embedding(x)

class PositionalEncoding(nn.Module):
    def __init__(self , context_length , d_model):
        super().__init__()
        self.embedding = nn.Embedding(context_length , d_model)
    def forward(self , x):
        B , T = x.shape
        positions = torch.arange(T , device = x.to(device))
        positions = positions.unsqueez(0).expand(B , T)
        return self.embedding(positions)

## Multi Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self , d_model , n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.head_dim = d_model // self.n_heads
        self.Wq = nn.Linear(d_model , d_model)
        self.Wk = nn.Linear(d_model , d_model)
        self.Wv = nn.Linear(d_model , d_model)
        self.out = nn.Linear(d_model , d_model)
    def forward(self , x):
        B ,T ,C = x.shape # Here X is 3 dimensioan because it has passed through the embedding and the positional encoding layer so it was B , T now it's B , T , C and C is d_model token
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)
        Q = torch.view(B, T, self.n_heads , self.head_dim).transpose(1,2)
        K = torch.view(B , T , self.n_heads , self.head_dim).transpose(1,2)
        V = torch.view(B , T , self.n_heads , self.head_dim).transpose(1,2)
        scores = torch.matmul(Q , K.transpose(2,1))
        scores = scores / math.sqrt(self.head_dim)
        mask = torch.tril(T, T)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        scores = F.softmax(scores , dim = -1)
        context = torch.matmul(scores , V)
        context = context.transpose(1,2).contiguous()
        context = context.view(B , T , C)
        output = self.out(context)
        return output

## Feed Forward Neural Network

In [ ]:
class FeedForwardNN(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.ff = nn.Sequential(
            nn.Linear(d_model , 4*d_model),
            nn.GELU(),
            nn.Linear(4*d_model , d_model)
        )
    def forward(self,x):
        return self.ff(x)

### Layer Normalization in Transoformer:
### layerNorm(x) = x - mean(x) / sqrt(var(x)+  epsilon)

In [ ]:
class Transformer_block(nn.Module):
    def __init__(self , d_model, n_heads):
        super().__init__()
        self.multi_head = MultiHeadAttention(d_model , n_heads)
        self.layerNorm = nn.LayerNorm(d_model)
        self.ff = FeedForwardNN(d_model)
        self.layerNorm2 = nn.LayerNorm(d_model)
    def forward(self , x):
        B , T , C = x.shape
        out = self.multi_head(x)
        out = self.ff(out)
        return out

## Mini GPT like parameters

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self , d_model , n_heads , context_length , vocab_size):
        super().__init__()
        self.embedding = Embedding(vocab_size , d_model)
        self.positions = PositionalEncoding(context_length , d_model)
        self.transformer = Transformer_block(d_model , n_heads)
        self.logits = nn.Linear(d_model , vocab_size) # this is for plicy
        self.value = nn.Linear(d_model , 1) # this is for the value head
    def forward(self , x):
        embedding = self.embedding(x)
        positions = self.positions(x)
        embedpositions = embedding + positions
        output = self.transformer(embedpositions)
        logits = self.logits(output)
        values = self.value(output)
        return logits , values # return policy and values policy os the distribution over the vocab and the value is the value oer token how much this token is helpful given what i've seen so far


In [ ]:
d_model = 768
n_heads = 12
context_length = 128
vocab_size = 50257

In [ ]:
model = MiniGPT(d_model , n_heads , context_length , vocab_size)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters() , lr = 3e-5)

## Get Log probabilities

#### this function is very useful it allows us to get the value of the chosen token (policy) based on actions array of indexes each index represents the chosen token and also the log_probs 3d array which the output of the model

In [6]:
def get_log_probs(logits , actions):
    log_probs = F.log_softmax(logits , dim=-1)
    log_probs = log_probs.gather(
        dim = -1,
        index = action.unsqueez(-1)
    ).squeez(-1)
    return log_probs

## PPO LOSS FUNCTION

#### This fucntion computes the loss of the PPO because there's no built is function for PPO because the PPO LOSS is contructed of three main losses policy loss + value loss + kl penalty to prevent big updates

In [8]:
def compute_ppo_loss(model , states , ations, old_log_probs , rewards , advantages , eps , value_coef,
                    kl_coef):
    logits , values = model(states)
    log_probs = get_log_probs(logits , actions)
    ratio = torch.exp(log_probs - old_log_probs) # in mathematics log(a)-log(b) = log(a/b)
    clipped = ratio*advantages # this is the policy loss
    unclipped = torch.clamp(ratio , 1-eps , 1+eps)*advantages # this is the unclipped policy loss
    policy_loss = -torch.min(clipped , unclipped).mean() # we take only the minimum
    value_loss = F.mse_loss(values - rewards) # this the loss of the value model values per token - reward distributed over token just copy the reward of one sequence on the tokens of this sequence
    kl = (old_log_probs - log_probs).mean() # this is to prevent big updates
    total_loss = policy_loss + value_loss*value_coef + kl*kl_coef # the loss formula
    return total_loss , policy_loss , value_loss , kl # return losses

In [2]:
import copy

## Training Loop

In [3]:
for epoch in range(epochs):
    old_model = copy.deepcopy(model)
    all_states = []
    all_actions = []
    all_reward = []
    all_old_log_robs = []
    for prompt in prompts:
        states = tokeniezr.decode(prompt) # the prompt is 128 because in my GPT-2 implementation the contet window is 128
        states = states.unsqueez(0) # (1,128) because that's what the model expects
        logits , values = old_model(states) # Logits.shape = (1,128,V) and values (1,128,1) one value per token
        old_probs = F.softmax(logits , dim = -1) # W eapply softmax to the vocabulmary pf each token
        actions = torch.multinomial(old_probs.view(-1 , old_probs.size(-1)),1) # now multinomial is a function that chooses the more likely next token not the highest one but the modt likely in this way it gives the model some creativity
        # actions.shape(1*128,50000) and it will chose for each line one single token by index so actions.shape = (128,1)
        actions = actions.view(states.shape) # now actions.shape = (1,128) [[id1 , id2 , id3,...]]
        old_log_probs = get_log_probs(logits , actions) # this return the log probabailities of the chosen toknes
        # old_log_probs.shape = (1,128)
        response = tokenizer.decode(actions[0].tolist()) # decode the response to feed it to the reward model with the question
        reward = reward_model(prompt , response) # get the reward for this sequence
        all_states.append(states)
        all_old_log_probs.append(old_probs)
        all_actions.append(actions)
        all_rewards.append(reward)
    states = torch.cat(all_states , dim=0) # now we have a tensor of states (B , T)
    old_log_probs = torch.cat(all_old_log_probs , dim = 0) # now we have a tensor of old_log_probs (B , T)
    actions = torch.cat(all_actions , dim = 0) # same we have a tensor of actions (B , T)
    rewards = torch.tensor(all_rewards).to(device) # now reward is shape (B , T) 
    rewards = torch.unsqueez(1).exapand_as(actions) #, (B , T , 128) for each token a value
    _ , values = model(states)
    for _ in range(ppo_steps):
        total_loss , policy_loss , value_loss , kl = compute_ppo_loss(model , states , actions , old_log_probs , rewards , advantages)
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
    print(f"Epoch {epoch} | Loss : {total_loss.item():.4f}")
        

NameError: name 'epochs' is not defined